In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import tensorflow as tf
import keras

train = pd.read_csv('/kaggle/input/forest-cover-type-prediction/train.csv')
test = pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv')

ids = test.pop('Id')

train = train.drop(['Id'], axis=1);
targets = keras.utils.to_categorical(train.pop('Cover_Type') - 1)

In [ ]:
train = train.fillna(0)
test = test.fillna(0)

In [ ]:
train['Distance'] = (train.pop('Horizontal_Distance_To_Hydrology') ** 2 +
                     train.pop('Vertical_Distance_To_Hydrology') ** 2) ** 0.2
test['Distance'] = (test.pop('Horizontal_Distance_To_Hydrology') ** 2 +
                     test.pop('Vertical_Distance_To_Hydrology') ** 2) ** 0.2

In [ ]:
train.describe()

In [ ]:
test.describe()

In [ ]:
normalize = ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Roadways', 
             'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm', 
             'Horizontal_Distance_To_Fire_Points', 'Distance']

In [ ]:
train = train.abs()
test = test.abs()

mean = train[normalize].mean(axis=0)
train[normalize] -= mean
std = train[normalize].std(axis=0)
train[normalize] /= std
test[normalize] -= mean
test[normalize] /= std

In [ ]:
train.describe()

In [ ]:
test.describe()

In [ ]:
model = keras.Sequential([keras.layers.Dense(128, activation='relu', input_dim=53),
                          keras.layers.Dense(128, activation='relu'),
                          keras.layers.Dense(128, activation='relu'),
                          keras.layers.Dense(7, activation='softmax')])

model.compile(optimizer="rmsprop",
              loss = 'categorical_crossentropy',
              metrics=['accuracy'])

model.fit(train, targets, epochs=1000, batch_size=1024, validation_split=0.0)

In [ ]:
predictions = model.predict(test, batch_size=128)
predictions = np.argmax(predictions, axis=1) + 1
display(predictions)

In [ ]:
output = pd.DataFrame({'Id': ids, 'Cover_Type': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")